# Naive Bayes génératif vs régression logistique discriminative (portage Rust)

**Navigation** : [<< 2.3 Régression linéaire et logistique](2.3-Regression-lineaire-logistique.ipynb) | [2.3c Régression en grande dimension >>](2.3c-Regression-Grande-Dimension.ipynb) | [Index 02-ML-Cours](README.md) | [Version Python](2.3b-Naive-Bayes-Generatif.ipynb) | [Version C#](2.3b-Naive-Bayes-Generatif-CSharp.ipynb)

Un classifieur peut modéliser le problème de **deux façons opposées**.

- La voie **générative** apprend la loi de chaque classe, $P(x \mid y)$, puis applique la règle de Bayes pour décider. Le **Naive Bayes gaussien** en est le représentant canonique : il suppose que, *conditionnellement à la classe*, les dimensions sont **indépendantes** et **gaussiennes**. L'hypothèse est presque toujours fausse — les dimensions d'un vrai jeu de données sont corrélées — et pourtant elle est souvent payante, parce qu'elle réduit le nombre de paramètres à estimer.
- La voie **discriminative** apprend directement $P(y \mid x)$, ou plus simplement une frontière de décision. La **régression logistique** en est le représentant canonique : aucune hypothèse sur la loi de $x$, une frontière linéaire apprise par maximum de vraisemblance.

**Fil conducteur.** Ce carnet écrit un Naive Bayes gaussien **de zéro** en Rust, le vérifie contre l'implémentation de `smartcore`, puis mesure les deux voies sur un protocole apparié : mêmes tirages, mêmes budgets d'entraînement, même jeu de test. La question n'est pas « lequel est le meilleur » — elle n'a pas de réponse dans l'absolu — mais **quel terrain donne l'avantage à laquelle, et à quel prix**.

## Le portage Rust

| Python (jumeau) | Rust (ce carnet) |
|---|---|
| `sklearn.naive_bayes.GaussianNB` | `smartcore::naive_bayes::gaussian::GaussianNB` |
| `sklearn.naive_bayes.MultinomialNB` | `smartcore::naive_bayes::multinomial::MultinomialNB` |
| `sklearn.linear_model.LogisticRegression(max_iter=8000, C=1e6)` | `LogisticRegression` + `LogisticRegressionParameters { solver: LBFGS, alpha: 1e-6 }` |
| `scipy.stats.norm.logpdf` | la log-densité écrite à la main |
| `numpy.random.default_rng(7)` | un générateur `XorShift64` + Box-Muller, écrit dans le carnet |
| `matplotlib` | une courbe en caractères |

Trois écarts sont assumés, et chacun a une raison.

1. **Aucune crate aléatoire.** Le générateur est écrit dans le carnet : une graine donne le même flux sur n'importe quelle machine, sans dépendre de la version d'une crate tierce.
2. **Les figures deviennent des cartes de caractères.** Une courbe `matplotlib` ne s'affiche pas sous `evcxr` ; une carte de caractères, deux colonnes par budget, montre la même chose — deux courbes, et l'écart entre elles.
3. **Les niveaux absolus ne sont pas ceux du jumeau Python.** Deux générateurs différents tirent deux populations différentes. Ce qui se compare d'un carnet à l'autre, ce sont les **régimes** (qui gagne, quand, et par quel mécanisme), et un registre de conformité vérifie que ces régimes concordent.

Chaque cellule de code redéclare les types et fonctions dont elle a besoin : une cellule doit pouvoir être rejouée seule, sans dépendre de l'ordre d'exécution.

In [2]:
// Configuration du jumeau Rust du carnet 2.3b : smartcore 0.7.0 suffit aux deux
// modeles compares (Naive Bayes gaussien et regression logistique).
// La dependance se declare ici, une seule fois : un :dep place plus bas
// recompile tout et efface les variables des cellules precedentes.
:dep smartcore = { version = "0.7.0", features = ["datasets"] }

println!("Configuration OK : 2.3b — Naive Bayes génératif contre régression logistique");

Configuration OK : 2.3b — Naive Bayes génératif contre régression logistique


## 1. Le formalisme, puis le code

Pour chaque classe $c$, Naive Bayes gaussien estime trois choses :

1. la **probabilité a priori** $\pi_c$ — la fréquence de la classe dans l'échantillon ;
2. la **moyenne** $\mu_{jc}$ de chaque dimension $j$ dans la classe ;
3. la **variance** $\sigma^2_{jc}$ de chaque dimension dans la classe.

L'hypothèse d'indépendance conditionnelle permet alors d'écrire la densité de la classe comme un produit :

$$P(x \mid c) = \prod_{j=1}^{d} \mathcal{N}(x_j ; \mu_{jc}, \sigma^2_{jc})$$

et la décision est l'argmax du score, pris en logarithme pour éviter le sous-débordement du produit :

$$\text{score}_c(x) = \log \pi_c + \sum_{j=1}^{d} \left[ -\tfrac{1}{2}\log(2\pi\sigma^2_{jc}) - \frac{(x_j - \mu_{jc})^2}{2\sigma^2_{jc}} \right]$$

Deux détails du code méritent d'être signalés.

- **Le logarithme n'est pas une coquetterie.** Pour $d = 12$ dimensions et une densité typique de l'ordre de $10^{-2}$, le produit tombe sous $10^{-24}$ : le logarithme est ce qui garde le calcul dans les nombres flottants.
- **L'epsilon de garde.** Une dimension constante dans une classe donne une variance nulle, donc une division par zéro. `scikit-learn` ajoute `var_smoothing * var(X).max()` à toutes les variances (par défaut $10^{-9}$) ; ce portage ajoute un **epsilon absolu** de $10^{-9}$, exactement pour la même raison. Les deux choix protègent la division, pas la qualité du modèle.

In [3]:
// Naive Bayes gaussien ecrit a la main.
// scipy.stats.norm.logpdf n'existe pas ici : la log-densite s'ecrit
//   log N(x ; mu, var) = -0.5 * ( log(2*pi*var) + (x - mu)^2 / var ).
struct MonGaussianNB {
    classes: Vec<u32>,
    moyens: Vec<Vec<f64>>,
    variances: Vec<Vec<f64>>,
    priors: Vec<f64>,
}

impl MonGaussianNB {
    fn fit(x: &[Vec<f64>], y: &[u32]) -> MonGaussianNB {
        let mut classes = y.to_vec();
        classes.sort_unstable();
        classes.dedup();
        let d = x[0].len();
        let n = x.len() as f64;
        let mut moyens = Vec::new();
        let mut variances = Vec::new();
        let mut priors = Vec::new();
        for &c in &classes {
            let lignes: Vec<&Vec<f64>> = x
                .iter()
                .zip(y.iter())
                .filter(|(_, yi)| **yi == c)
                .map(|(xi, _)| xi)
                .collect();
            let m = lignes.len() as f64;
            let mut mu = vec![0.0; d];
            for ligne in &lignes {
                for j in 0..d {
                    mu[j] += ligne[j];
                }
            }
            for j in 0..d {
                mu[j] /= m;
            }
            // Variance de population, puis l'epsilon de garde : sans lui une
            // dimension constante annule la densite (division par zero).
            let mut var = vec![0.0; d];
            for ligne in &lignes {
                for j in 0..d {
                    var[j] += (ligne[j] - mu[j]).powi(2);
                }
            }
            for j in 0..d {
                var[j] = var[j] / m + 1e-9;
            }
            moyens.push(mu);
            variances.push(var);
            priors.push(m / n);
        }
        MonGaussianNB { classes, moyens, variances, priors }
    }

    // Un score par classe et par ligne : log(prior) + somme des log-densites.
    fn log_vraisemblance(&self, x: &[Vec<f64>]) -> Vec<Vec<f64>> {
        x.iter()
            .map(|ligne| {
                self.classes
                    .iter()
                    .enumerate()
                    .map(|(k, _)| {
                        let mut score = self.priors[k].ln();
                        for j in 0..ligne.len() {
                            let var = self.variances[k][j];
                            let ecart = ligne[j] - self.moyens[k][j];
                            score += -0.5
                                * ((2.0 * std::f64::consts::PI * var).ln() + ecart * ecart / var);
                        }
                        score
                    })
                    .collect()
            })
            .collect()
    }

    fn predict(&self, x: &[Vec<f64>]) -> Vec<u32> {
        self.log_vraisemblance(x)
            .iter()
            .map(|scores| {
                let mut meilleur = 0usize;
                for k in 1..scores.len() {
                    if scores[k] > scores[meilleur] {
                        meilleur = k;
                    }
                }
                self.classes[meilleur]
            })
            .collect()
    }
}

println!("Naive Bayes gaussien from scratch prêt.");

Naive Bayes gaussien from scratch prêt.


In [4]:
// Verification : le modele ecrit a la main et celui de smartcore recoivent
// exactement les memes donnees, et doivent rendre les memes predictions.
use smartcore::linalg::basic::matrix::DenseMatrix;
use smartcore::naive_bayes::gaussian::GaussianNB;

// Naive Bayes gaussien ecrit a la main.
// scipy.stats.norm.logpdf n'existe pas ici : la log-densite s'ecrit
//   log N(x ; mu, var) = -0.5 * ( log(2*pi*var) + (x - mu)^2 / var ).
struct MonGaussianNB {
    classes: Vec<u32>,
    moyens: Vec<Vec<f64>>,
    variances: Vec<Vec<f64>>,
    priors: Vec<f64>,
}

impl MonGaussianNB {
    fn fit(x: &[Vec<f64>], y: &[u32]) -> MonGaussianNB {
        let mut classes = y.to_vec();
        classes.sort_unstable();
        classes.dedup();
        let d = x[0].len();
        let n = x.len() as f64;
        let mut moyens = Vec::new();
        let mut variances = Vec::new();
        let mut priors = Vec::new();
        for &c in &classes {
            let lignes: Vec<&Vec<f64>> = x
                .iter()
                .zip(y.iter())
                .filter(|(_, yi)| **yi == c)
                .map(|(xi, _)| xi)
                .collect();
            let m = lignes.len() as f64;
            let mut mu = vec![0.0; d];
            for ligne in &lignes {
                for j in 0..d {
                    mu[j] += ligne[j];
                }
            }
            for j in 0..d {
                mu[j] /= m;
            }
            // Variance de population, puis l'epsilon de garde : sans lui une
            // dimension constante annule la densite (division par zero).
            let mut var = vec![0.0; d];
            for ligne in &lignes {
                for j in 0..d {
                    var[j] += (ligne[j] - mu[j]).powi(2);
                }
            }
            for j in 0..d {
                var[j] = var[j] / m + 1e-9;
            }
            moyens.push(mu);
            variances.push(var);
            priors.push(m / n);
        }
        MonGaussianNB { classes, moyens, variances, priors }
    }

    // Un score par classe et par ligne : log(prior) + somme des log-densites.
    fn log_vraisemblance(&self, x: &[Vec<f64>]) -> Vec<Vec<f64>> {
        x.iter()
            .map(|ligne| {
                self.classes
                    .iter()
                    .enumerate()
                    .map(|(k, _)| {
                        let mut score = self.priors[k].ln();
                        for j in 0..ligne.len() {
                            let var = self.variances[k][j];
                            let ecart = ligne[j] - self.moyens[k][j];
                            score += -0.5
                                * ((2.0 * std::f64::consts::PI * var).ln() + ecart * ecart / var);
                        }
                        score
                    })
                    .collect()
            })
            .collect()
    }

    fn predict(&self, x: &[Vec<f64>]) -> Vec<u32> {
        self.log_vraisemblance(x)
            .iter()
            .map(|scores| {
                let mut meilleur = 0usize;
                for k in 1..scores.len() {
                    if scores[k] > scores[meilleur] {
                        meilleur = k;
                    }
                }
                self.classes[meilleur]
            })
            .collect()
    }
}

// Generateur maison : xorshift64 pour les entiers, Box-Muller pour le normal.
// Aucune crate `rand` : le flux est reproductible par graine, et identique d'une
// cellule a l'autre sans dependre d'une version tierce.
struct XorShift64 {
    etat: u64,
}

impl XorShift64 {
    fn new(graine: u64) -> XorShift64 {
        XorShift64 {
            etat: if graine == 0 { 0x9E3779B97F4A7C15 } else { graine },
        }
    }

    fn suivant(&mut self) -> u64 {
        let mut x = self.etat;
        x ^= x << 13;
        x ^= x >> 7;
        x ^= x << 17;
        self.etat = x;
        x
    }

    // Uniforme sur [0, 1) : 53 bits de mantisse, comme numpy.
    fn uniforme(&mut self) -> f64 {
        (self.suivant() >> 11) as f64 / ((1u64 << 53) as f64)
    }

    // Normale centree reduite par Box-Muller.
    fn normal(&mut self) -> f64 {
        let u1 = self.uniforme().max(1e-12);
        let u2 = self.uniforme();
        (-2.0 * u1.ln()).sqrt() * (2.0 * std::f64::consts::PI * u2).cos()
    }

    fn entier(&mut self, borne: u64) -> u64 {
        self.suivant() % borne
    }
}

fn precision(y_vrai: &[u32], y_pred: &[u32]) -> f64 {
    let bons = y_vrai
        .iter()
        .zip(y_pred.iter())
        .filter(|(a, b)| a == b)
        .count();
    bons as f64 / y_vrai.len() as f64
}

let d = 6usize;
let n = 200usize;
let mut rng = XorShift64::new(7);
// Deux classes gaussiennes centrees reduites, la seconde decalee de 1.2.
let mut x: Vec<Vec<f64>> = (0..n).map(|_| (0..d).map(|_| rng.normal()).collect()).collect();
let y: Vec<u32> = (0..n).map(|_| rng.entier(2) as u32).collect();
for i in 0..n {
    if y[i] == 1 {
        for j in 0..d {
            x[i][j] += 1.2;
        }
    }
}

let maison = MonGaussianNB::fit(&x, &y);
let pred_maison = maison.predict(&x);

let matrice = DenseMatrix::from_2d_vec(&x).unwrap();
let reference = GaussianNB::fit(&matrice, &y, Default::default()).unwrap();
let pred_reference = reference.predict(&matrice).unwrap();

println!("accuracy from scratch : {:.3}", precision(&y, &pred_maison));
println!("accuracy smartcore    : {:.3}", precision(&y, &pred_reference));
println!("predictions identiques (0/1) : {}", pred_maison == pred_reference);

let moyenne_classe1: Vec<f64> = (0..3)
    .map(|j| {
        let indices: Vec<usize> = (0..n).filter(|&i| y[i] == 1).collect();
        indices.iter().map(|&i| x[i][j]).sum::<f64>() / indices.len() as f64
    })
    .collect();
println!(
    "moyenne empirique classe1, 3 premieres dims : [{:.3}, {:.3}, {:.3}]",
    moyenne_classe1[0], moyenne_classe1[1], moyenne_classe1[2]
);

accuracy from scratch : 0.965


accuracy smartcore    : 0.965


predictions identiques (0/1) : true


moyenne empirique classe1, 3 premieres dims : [1.080, 1.181, 1.237]


### Lecture du résultat

Les deux modèles rendent la **même précision** et les **mêmes prédictions** : l'implémentation écrite à la main retrouve celle de la bibliothèque.

C'est le premier contrôle de ce carnet, et il porte sur le code, pas sur la théorie : si l'epsilon de garde, la variance de population (`/n`, et non `/n-1`) ou l'ordre des classes divergeaient, les deux colonnes de prédictions se sépareraient sur quelques points. Elles ne se séparent pas.

La moyenne empirique de la classe 1 reste entre 1,08 et 1,24 selon la dimension, autour de la valeur 1,2 du décalage ajouté : c'est ce décalage qui rend le problème séparable à 96,5 %. Notez que `smartcore` n'expose **aucun accesseur** sur les moyennes et variances apprises — la vérification porte donc sur les prédictions, seule grandeur observable des deux côtés.

## Exercice 1 — les probabilités multinomiales à la main

Le Naive Bayes gaussien suppose des dimensions continues. Le **multinomial** suppose des **comptes** (des occurrences de mots, par exemple) et estime :

$$\theta_{j \mid c} = \frac{\text{nombre d'occurrences de } j \text{ dans la classe } c}{\text{total des occurrences de la classe } c}$$

Complétez `probabilites_multinomiales` : pour chaque classe, dans l'ordre croissant, elle renvoie la classe, le vecteur $\theta_{\cdot \mid c}$ et l'a priori $\pi_c$. Les deux jeux de test sont des tables de comptes en petits entiers.

Le carnet vérifie trois choses : que chaque vecteur $\theta$ somme à 1, que chaque a priori égale l'effectif relatif de sa classe, et que l'argmax du score multinomial coïncide avec les prédictions de `smartcore::naive_bayes::multinomial::MultinomialNB` (avec lissage de Laplace, $\alpha = 1$).

In [5]:
// Exercice 1 : probabilites multinomiales et a priori, par maximum de vraisemblance.
use smartcore::linalg::basic::matrix::DenseMatrix;
use smartcore::naive_bayes::multinomial::{MultinomialNB, MultinomialNBParameters};

/// Pour chaque classe dans l'ordre croissant : (classe, theta par dimension, a priori).
fn probabilites_multinomiales(x: &[Vec<f64>], y: &[u32]) -> Option<Vec<(u32, Vec<f64>, f64)>> {
    // TODO : sommer les comptes par classe, diviser par le total de la classe
    // pour obtenir theta, et prendre l'effectif relatif comme a priori.
    // Indice : aucun lissage ici, c'est le maximum de vraisemblance brut.
    None
}

let fixtures: Vec<(Vec<Vec<f64>>, Vec<u32>)> = vec![
    (
        vec![
            vec![2.0, 1.0, 0.0],
            vec![0.0, 3.0, 1.0],
            vec![1.0, 0.0, 2.0],
            vec![0.0, 1.0, 3.0],
        ],
        vec![0, 1, 0, 1],
    ),
    (
        vec![vec![3.0, 0.0], vec![1.0, 2.0], vec![0.0, 1.0]],
        vec![0, 0, 1],
    ),
];

let reponses: Vec<Option<Vec<(u32, Vec<f64>, f64)>>> = fixtures
    .iter()
    .map(|(xi, yi)| probabilites_multinomiales(xi, yi))
    .collect();

if reponses.iter().all(|r| r.is_none()) {
    println!("Exercice 1 à compléter.");
} else if reponses.iter().any(|r| r.is_none()) {
    println!("Exercice 1 : à revoir");
} else {
    let mut conforme = true;
    for ((xi, yi), reponse) in fixtures.iter().zip(reponses.iter()) {
        let resultat = reponse.as_ref().unwrap();
        for (classe, theta, prior) in resultat {
            let somme: f64 = theta.iter().sum();
            conforme = conforme && (somme - 1.0).abs() < 1e-9;
            let effectif = yi.iter().filter(|c| *c == classe).count() as f64 / yi.len() as f64;
            conforme = conforme && (prior - effectif).abs() < 1e-12;
        }
        // Contre-epreuve : smartcore ajuste le meme modele multinomial (alpha = 1)
        // et ses predictions doivent coincider avec l'argmax des theta etudiants.
        let comptes: Vec<Vec<u32>> = xi
            .iter()
            .map(|ligne| ligne.iter().map(|v| *v as u32).collect())
            .collect();
        let matrice = DenseMatrix::from_2d_vec(&comptes).unwrap();
        let modele = MultinomialNB::fit(
            &matrice,
            yi,
            MultinomialNBParameters { alpha: 1.0, priors: None },
        )
        .unwrap();
        let pred = modele.predict(&matrice).unwrap();
        let mut divergences = 0;
        for (i, ligne) in xi.iter().enumerate() {
            let mut meilleur = 0usize;
            let mut meilleur_score = f64::NEG_INFINITY;
            for (k, (_, theta, prior)) in resultat.iter().enumerate() {
                let mut score = prior.ln();
                for j in 0..ligne.len() {
                    score += ligne[j] * theta[j].max(1e-12).ln();
                }
                if score > meilleur_score {
                    meilleur_score = score;
                    meilleur = k;
                }
            }
            if resultat[meilleur].0 != pred[i] {
                divergences += 1;
            }
        }
        conforme = conforme && divergences == 0;
    }
    if conforme {
        println!("Exercice 1 : calculs conformes ; predictions identiques a smartcore MultinomialNB");
    } else {
        println!("Exercice 1 : à revoir");
    }
};

Exercice 1 à compléter.


### Guide de l'exercice 1

Le maximum de vraisemblance multinomial ne demande qu'une accumulation :

1. trier les classes de `y` (comme `MonGaussianNB::fit` le fait), pour que l'ordre soit déterministe ;
2. pour chaque classe, sommer les comptes de chaque dimension sur les lignes de cette classe — un accumulateur `Vec<f64>` de la longueur d'une ligne ;
3. diviser ce vecteur par le total de la classe (la somme de toutes ses cases) : c'est $\theta_{\cdot \mid c}$ ;
4. l'a priori est l'effectif de la classe divisé par le nombre de lignes.

Deux pièges, tous deux vérifiés par le carnet :

- **La normalisation porte sur le total de la classe**, pas sur le nombre de lignes. Une classe dont les documents sont longs a un grand total ; diviser par le nombre de lignes rendrait $\theta$ faux et la somme différente de 1.
- **Aucun lissage ici.** Le maximum de vraisemblance brut donne $\theta_j = 0$ pour une dimension jamais observée. C'est pourquoi la contre-épreuve utilise `alpha = 1` : `smartcore` lisse, votre fonction non, et sur ces deux tables aucun compte nul n'apparaît — l'argmax coïncide donc. Sur un vrai corpus de texte, l'absence de lissage rendrait le score infini dès qu'un mot inconnu apparaît.

## 2. Le protocole apparié : huit budgets, quarante tirages

Comparer deux classifieurs sur un seul découpage ne dit rien : la variance d'échantillonnage domine. Le protocole est donc apparié.

- **Huit budgets d'entraînement** : $n \in \{15, 30, 60, 120, 240, 480, 960, 1920\}$.
- **Quarante tirages par budget**, chacun avec sa graine dérivée de la graine du scénario (deux flux indépendants par graine : l'un pour l'entraînement, l'autre pour le test).
- **Un jeu de test disjoint**, de taille $2 n_{\max} - n$ : à $n = 15$ il porte 3 825 points, à $n = 1920$ encore 1 920. La précision se compare donc à mieux que le millième près.
- **Les deux modèles voient exactement les mêmes données**, dans le même ordre.

Deux terrains, deux hypothèses fausses pour Naive Bayes, de deux façons différentes.

- **Scénario A — corrélations homogènes** ($\rho = 0{,}6$). Un seul facteur latent porte le signal, toutes les dimensions en dérivent avec la même corrélation. L'hypothèse d'indépendance est fausse, mais elle est fausse *uniformément* : elle ne biaise pas la direction de décision, elle gonfle seulement la confiance.
- **Scénario B — corrélations en bloc** ($\rho = 0{,}85$). Deux blocs de six dimensions suivent chacun leur facteur latent, le signal est la **différence** des deux, et l'échelle des dimensions varie d'un facteur huit à l'intérieur de chaque bloc. Ici l'hypothèse d'indépendance ne gonfle pas seulement la confiance : elle **désaligne** la direction de décision, parce que Naive Bayes pondère chaque dimension par sa variance marginale alors que l'optimum dépend de la matrice de covariance jointe.

Ce que Prong B de la politique SOTA demande est tenu ici : le terrain n'est pas dégénéré — sur un problème où les dimensions seraient indépendantes, les deux modèles convergeraient vers la même solution et le carnet n'aurait rien à montrer.

In [6]:
use std::time::Instant;
use smartcore::linalg::basic::matrix::DenseMatrix;
use smartcore::linear::logistic_regression::{
    LogisticRegression, LogisticRegressionParameters, LogisticRegressionSolverName,
};

// Generateur maison : xorshift64 pour les entiers, Box-Muller pour le normal.
// Aucune crate `rand` : le flux est reproductible par graine, et identique d'une
// cellule a l'autre sans dependre d'une version tierce.
struct XorShift64 {
    etat: u64,
}

impl XorShift64 {
    fn new(graine: u64) -> XorShift64 {
        XorShift64 {
            etat: if graine == 0 { 0x9E3779B97F4A7C15 } else { graine },
        }
    }

    fn suivant(&mut self) -> u64 {
        let mut x = self.etat;
        x ^= x << 13;
        x ^= x >> 7;
        x ^= x << 17;
        self.etat = x;
        x
    }

    // Uniforme sur [0, 1) : 53 bits de mantisse, comme numpy.
    fn uniforme(&mut self) -> f64 {
        (self.suivant() >> 11) as f64 / ((1u64 << 53) as f64)
    }

    // Normale centree reduite par Box-Muller.
    fn normal(&mut self) -> f64 {
        let u1 = self.uniforme().max(1e-12);
        let u2 = self.uniforme();
        (-2.0 * u1.ln()).sqrt() * (2.0 * std::f64::consts::PI * u2).cos()
    }

    fn entier(&mut self, borne: u64) -> u64 {
        self.suivant() % borne
    }
}

fn sigmoide(z: f64) -> f64 {
    1.0 / (1.0 + (-z).exp())
}

fn precision(y_vrai: &[u32], y_pred: &[u32]) -> f64 {
    let bons = y_vrai
        .iter()
        .zip(y_pred.iter())
        .filter(|(a, b)| a == b)
        .count();
    bons as f64 / y_vrai.len() as f64
}

// Naive Bayes gaussien ecrit a la main.
// scipy.stats.norm.logpdf n'existe pas ici : la log-densite s'ecrit
//   log N(x ; mu, var) = -0.5 * ( log(2*pi*var) + (x - mu)^2 / var ).
struct MonGaussianNB {
    classes: Vec<u32>,
    moyens: Vec<Vec<f64>>,
    variances: Vec<Vec<f64>>,
    priors: Vec<f64>,
}

impl MonGaussianNB {
    fn fit(x: &[Vec<f64>], y: &[u32]) -> MonGaussianNB {
        let mut classes = y.to_vec();
        classes.sort_unstable();
        classes.dedup();
        let d = x[0].len();
        let n = x.len() as f64;
        let mut moyens = Vec::new();
        let mut variances = Vec::new();
        let mut priors = Vec::new();
        for &c in &classes {
            let lignes: Vec<&Vec<f64>> = x
                .iter()
                .zip(y.iter())
                .filter(|(_, yi)| **yi == c)
                .map(|(xi, _)| xi)
                .collect();
            let m = lignes.len() as f64;
            let mut mu = vec![0.0; d];
            for ligne in &lignes {
                for j in 0..d {
                    mu[j] += ligne[j];
                }
            }
            for j in 0..d {
                mu[j] /= m;
            }
            // Variance de population, puis l'epsilon de garde : sans lui une
            // dimension constante annule la densite (division par zero).
            let mut var = vec![0.0; d];
            for ligne in &lignes {
                for j in 0..d {
                    var[j] += (ligne[j] - mu[j]).powi(2);
                }
            }
            for j in 0..d {
                var[j] = var[j] / m + 1e-9;
            }
            moyens.push(mu);
            variances.push(var);
            priors.push(m / n);
        }
        MonGaussianNB { classes, moyens, variances, priors }
    }

    // Un score par classe et par ligne : log(prior) + somme des log-densites.
    fn log_vraisemblance(&self, x: &[Vec<f64>]) -> Vec<Vec<f64>> {
        x.iter()
            .map(|ligne| {
                self.classes
                    .iter()
                    .enumerate()
                    .map(|(k, _)| {
                        let mut score = self.priors[k].ln();
                        for j in 0..ligne.len() {
                            let var = self.variances[k][j];
                            let ecart = ligne[j] - self.moyens[k][j];
                            score += -0.5
                                * ((2.0 * std::f64::consts::PI * var).ln() + ecart * ecart / var);
                        }
                        score
                    })
                    .collect()
            })
            .collect()
    }

    fn predict(&self, x: &[Vec<f64>]) -> Vec<u32> {
        self.log_vraisemblance(x)
            .iter()
            .map(|scores| {
                let mut meilleur = 0usize;
                for k in 1..scores.len() {
                    if scores[k] > scores[meilleur] {
                        meilleur = k;
                    }
                }
                self.classes[meilleur]
            })
            .collect()
    }
}

// Scenario A -- correlations homogenes : un seul facteur latent porte le signal,
// toutes les dimensions en derivent avec le meme rho. C'est le terrain ou
// l'hypothese d'independance de Naive Bayes est fausse mais peu couteuse.
fn make_corr(n: usize, d: usize, decalage: f64, rho: f64, rng: &mut XorShift64) -> (Vec<Vec<f64>>, Vec<u32>) {
    let mut x = Vec::with_capacity(n);
    let mut y = Vec::with_capacity(n);
    for _ in 0..n {
        let facteur = rng.normal();
        let mut ligne = Vec::with_capacity(d);
        for _ in 0..d {
            let e = rng.normal();
            ligne.push(rho.sqrt() * facteur + (1.0 - rho).sqrt() * e);
        }
        let score: f64 = ligne.iter().take(d / 2).sum::<f64>() / ((d / 2) as f64).sqrt();
        let classe = if rng.uniforme() < sigmoide(score) { 1u32 } else { 0u32 };
        if classe == 1 {
            let poussee = decalage / (d as f64).sqrt();
            for v in ligne.iter_mut() {
                *v += poussee;
            }
        }
        x.push(ligne);
        y.push(classe);
    }
    (x, y)
}

// Scenario B -- correlations en bloc, et deux pieges pour l'hypothese
// d'independance. Les deux moities de dimensions suivent chacune son facteur
// latent, et le signal est la DIFFERENCE des deux ; surtout, l'echelle des
// dimensions varie d'un facteur huit a l'interieur de chaque bloc, alors que le
// decalage de classe est le meme en valeur absolue sur toutes. Naive Bayes lit
// chaque dimension separement et pondere donc de travers ; la regression
// logistique, elle, peut apprendre la direction jointe.
fn make_block(n: usize, d: usize, decalage: f64, rho: f64, rng: &mut XorShift64) -> (Vec<Vec<f64>>, Vec<u32>) {
    let mut x = Vec::with_capacity(n);
    let mut y = Vec::with_capacity(n);
    let moitie = d / 2;
    for _ in 0..n {
        let facteur_a = rng.normal();
        let facteur_b = rng.normal();
        let mut ligne = Vec::with_capacity(d);
        for j in 0..d {
            let e = rng.normal();
            let (facteur, k) = if j < moitie { (facteur_a, j) } else { (facteur_b, j - moitie) };
            let echelle = 2f64.powi(3 - (k as i32));
            ligne.push(echelle * (rho.sqrt() * facteur + (1.0 - rho).sqrt() * e));
        }
        let score = 4.0 * decalage * (facteur_a - facteur_b) / std::f64::consts::SQRT_2;
        let classe = if rng.uniforme() < sigmoide(score) { 1u32 } else { 0u32 };
        if classe == 1 {
            let poussee = decalage / (d as f64).sqrt();
            for v in ligne.iter_mut() {
                *v += poussee;
            }
        }
        x.push(ligne);
        y.push(classe);
    }
    (x, y)
}

// Protocole apparie : pour chaque graine, un seul tirage de taille n_max, dont
// les n premieres lignes servent d'entrainement et le reste de test. Le jeu de
// test est donc disjoint et de taille 2*n_max - n au minimum.
fn eval_curve(
    make: fn(usize, usize, f64, f64, &mut XorShift64) -> (Vec<Vec<f64>>, Vec<u32>),
    d: usize,
    decalage: f64,
    rho: f64,
    ns: &[usize],
    n_graines: usize,
    graine: u64,
) -> Vec<(f64, f64)> {
    let n_max = ns.iter().cloned().max().unwrap() * 2;
    let mut acc_nb = vec![0.0; ns.len()];
    let mut acc_lr = vec![0.0; ns.len()];
    for g in 0..n_graines {
        let mut rng = XorShift64::new(graine * 1000 + g as u64 + 1);
        let (x, y) = make(n_max, d, decalage, rho, &mut rng);
        for (k, &n) in ns.iter().enumerate() {
            let x_train = x[..n].to_vec();
            let y_train = y[..n].to_vec();
            let x_test = x[n..].to_vec();
            let y_test = y[n..].to_vec();

            let nb = MonGaussianNB::fit(&x_train, &y_train);
            acc_nb[k] += precision(&y_test, &nb.predict(&x_test));

            // alpha = 1e-6 joue le role du C = 1e6 du jumeau Python : la
            // regularisation est rendue negligeable devant la vraisemblance.
            let matrice = DenseMatrix::from_2d_vec(&x_train).unwrap();
            let modele = LogisticRegression::fit(
                &matrice,
                &y_train,
                LogisticRegressionParameters {
                    solver: LogisticRegressionSolverName::LBFGS,
                    alpha: 1e-6,
                },
            )
            .unwrap();
            let matrice_test = DenseMatrix::from_2d_vec(&x_test).unwrap();
            acc_lr[k] += precision(&y_test, &modele.predict(&matrice_test).unwrap());
        }
    }
    ns.iter()
        .enumerate()
        .map(|(k, _)| (acc_nb[k] / n_graines as f64, acc_lr[k] / n_graines as f64))
        .collect()
}

println!("Générateurs et protocole de 40 tirages prêts.");

Générateurs et protocole de 40 tirages prêts.


In [7]:
use std::time::Instant;
use smartcore::linalg::basic::matrix::DenseMatrix;
use smartcore::linear::logistic_regression::{
    LogisticRegression, LogisticRegressionParameters, LogisticRegressionSolverName,
};

// Generateur maison : xorshift64 pour les entiers, Box-Muller pour le normal.
// Aucune crate `rand` : le flux est reproductible par graine, et identique d'une
// cellule a l'autre sans dependre d'une version tierce.
struct XorShift64 {
    etat: u64,
}

impl XorShift64 {
    fn new(graine: u64) -> XorShift64 {
        XorShift64 {
            etat: if graine == 0 { 0x9E3779B97F4A7C15 } else { graine },
        }
    }

    fn suivant(&mut self) -> u64 {
        let mut x = self.etat;
        x ^= x << 13;
        x ^= x >> 7;
        x ^= x << 17;
        self.etat = x;
        x
    }

    // Uniforme sur [0, 1) : 53 bits de mantisse, comme numpy.
    fn uniforme(&mut self) -> f64 {
        (self.suivant() >> 11) as f64 / ((1u64 << 53) as f64)
    }

    // Normale centree reduite par Box-Muller.
    fn normal(&mut self) -> f64 {
        let u1 = self.uniforme().max(1e-12);
        let u2 = self.uniforme();
        (-2.0 * u1.ln()).sqrt() * (2.0 * std::f64::consts::PI * u2).cos()
    }

    fn entier(&mut self, borne: u64) -> u64 {
        self.suivant() % borne
    }
}

fn sigmoide(z: f64) -> f64 {
    1.0 / (1.0 + (-z).exp())
}

fn precision(y_vrai: &[u32], y_pred: &[u32]) -> f64 {
    let bons = y_vrai
        .iter()
        .zip(y_pred.iter())
        .filter(|(a, b)| a == b)
        .count();
    bons as f64 / y_vrai.len() as f64
}

// Naive Bayes gaussien ecrit a la main.
// scipy.stats.norm.logpdf n'existe pas ici : la log-densite s'ecrit
//   log N(x ; mu, var) = -0.5 * ( log(2*pi*var) + (x - mu)^2 / var ).
struct MonGaussianNB {
    classes: Vec<u32>,
    moyens: Vec<Vec<f64>>,
    variances: Vec<Vec<f64>>,
    priors: Vec<f64>,
}

impl MonGaussianNB {
    fn fit(x: &[Vec<f64>], y: &[u32]) -> MonGaussianNB {
        let mut classes = y.to_vec();
        classes.sort_unstable();
        classes.dedup();
        let d = x[0].len();
        let n = x.len() as f64;
        let mut moyens = Vec::new();
        let mut variances = Vec::new();
        let mut priors = Vec::new();
        for &c in &classes {
            let lignes: Vec<&Vec<f64>> = x
                .iter()
                .zip(y.iter())
                .filter(|(_, yi)| **yi == c)
                .map(|(xi, _)| xi)
                .collect();
            let m = lignes.len() as f64;
            let mut mu = vec![0.0; d];
            for ligne in &lignes {
                for j in 0..d {
                    mu[j] += ligne[j];
                }
            }
            for j in 0..d {
                mu[j] /= m;
            }
            // Variance de population, puis l'epsilon de garde : sans lui une
            // dimension constante annule la densite (division par zero).
            let mut var = vec![0.0; d];
            for ligne in &lignes {
                for j in 0..d {
                    var[j] += (ligne[j] - mu[j]).powi(2);
                }
            }
            for j in 0..d {
                var[j] = var[j] / m + 1e-9;
            }
            moyens.push(mu);
            variances.push(var);
            priors.push(m / n);
        }
        MonGaussianNB { classes, moyens, variances, priors }
    }

    // Un score par classe et par ligne : log(prior) + somme des log-densites.
    fn log_vraisemblance(&self, x: &[Vec<f64>]) -> Vec<Vec<f64>> {
        x.iter()
            .map(|ligne| {
                self.classes
                    .iter()
                    .enumerate()
                    .map(|(k, _)| {
                        let mut score = self.priors[k].ln();
                        for j in 0..ligne.len() {
                            let var = self.variances[k][j];
                            let ecart = ligne[j] - self.moyens[k][j];
                            score += -0.5
                                * ((2.0 * std::f64::consts::PI * var).ln() + ecart * ecart / var);
                        }
                        score
                    })
                    .collect()
            })
            .collect()
    }

    fn predict(&self, x: &[Vec<f64>]) -> Vec<u32> {
        self.log_vraisemblance(x)
            .iter()
            .map(|scores| {
                let mut meilleur = 0usize;
                for k in 1..scores.len() {
                    if scores[k] > scores[meilleur] {
                        meilleur = k;
                    }
                }
                self.classes[meilleur]
            })
            .collect()
    }
}

// Scenario A -- correlations homogenes : un seul facteur latent porte le signal,
// toutes les dimensions en derivent avec le meme rho. C'est le terrain ou
// l'hypothese d'independance de Naive Bayes est fausse mais peu couteuse.
fn make_corr(n: usize, d: usize, decalage: f64, rho: f64, rng: &mut XorShift64) -> (Vec<Vec<f64>>, Vec<u32>) {
    let mut x = Vec::with_capacity(n);
    let mut y = Vec::with_capacity(n);
    for _ in 0..n {
        let facteur = rng.normal();
        let mut ligne = Vec::with_capacity(d);
        for _ in 0..d {
            let e = rng.normal();
            ligne.push(rho.sqrt() * facteur + (1.0 - rho).sqrt() * e);
        }
        let score: f64 = ligne.iter().take(d / 2).sum::<f64>() / ((d / 2) as f64).sqrt();
        let classe = if rng.uniforme() < sigmoide(score) { 1u32 } else { 0u32 };
        if classe == 1 {
            let poussee = decalage / (d as f64).sqrt();
            for v in ligne.iter_mut() {
                *v += poussee;
            }
        }
        x.push(ligne);
        y.push(classe);
    }
    (x, y)
}

// Scenario B -- correlations en bloc, et deux pieges pour l'hypothese
// d'independance. Les deux moities de dimensions suivent chacune son facteur
// latent, et le signal est la DIFFERENCE des deux ; surtout, l'echelle des
// dimensions varie d'un facteur huit a l'interieur de chaque bloc, alors que le
// decalage de classe est le meme en valeur absolue sur toutes. Naive Bayes lit
// chaque dimension separement et pondere donc de travers ; la regression
// logistique, elle, peut apprendre la direction jointe.
fn make_block(n: usize, d: usize, decalage: f64, rho: f64, rng: &mut XorShift64) -> (Vec<Vec<f64>>, Vec<u32>) {
    let mut x = Vec::with_capacity(n);
    let mut y = Vec::with_capacity(n);
    let moitie = d / 2;
    for _ in 0..n {
        let facteur_a = rng.normal();
        let facteur_b = rng.normal();
        let mut ligne = Vec::with_capacity(d);
        for j in 0..d {
            let e = rng.normal();
            let (facteur, k) = if j < moitie { (facteur_a, j) } else { (facteur_b, j - moitie) };
            let echelle = 2f64.powi(3 - (k as i32));
            ligne.push(echelle * (rho.sqrt() * facteur + (1.0 - rho).sqrt() * e));
        }
        let score = 4.0 * decalage * (facteur_a - facteur_b) / std::f64::consts::SQRT_2;
        let classe = if rng.uniforme() < sigmoide(score) { 1u32 } else { 0u32 };
        if classe == 1 {
            let poussee = decalage / (d as f64).sqrt();
            for v in ligne.iter_mut() {
                *v += poussee;
            }
        }
        x.push(ligne);
        y.push(classe);
    }
    (x, y)
}

// Protocole apparie : pour chaque graine, un seul tirage de taille n_max, dont
// les n premieres lignes servent d'entrainement et le reste de test. Le jeu de
// test est donc disjoint et de taille 2*n_max - n au minimum.
fn eval_curve(
    make: fn(usize, usize, f64, f64, &mut XorShift64) -> (Vec<Vec<f64>>, Vec<u32>),
    d: usize,
    decalage: f64,
    rho: f64,
    ns: &[usize],
    n_graines: usize,
    graine: u64,
) -> Vec<(f64, f64)> {
    let n_max = ns.iter().cloned().max().unwrap() * 2;
    let mut acc_nb = vec![0.0; ns.len()];
    let mut acc_lr = vec![0.0; ns.len()];
    for g in 0..n_graines {
        let mut rng = XorShift64::new(graine * 1000 + g as u64 + 1);
        let (x, y) = make(n_max, d, decalage, rho, &mut rng);
        for (k, &n) in ns.iter().enumerate() {
            let x_train = x[..n].to_vec();
            let y_train = y[..n].to_vec();
            let x_test = x[n..].to_vec();
            let y_test = y[n..].to_vec();

            let nb = MonGaussianNB::fit(&x_train, &y_train);
            acc_nb[k] += precision(&y_test, &nb.predict(&x_test));

            // alpha = 1e-6 joue le role du C = 1e6 du jumeau Python : la
            // regularisation est rendue negligeable devant la vraisemblance.
            let matrice = DenseMatrix::from_2d_vec(&x_train).unwrap();
            let modele = LogisticRegression::fit(
                &matrice,
                &y_train,
                LogisticRegressionParameters {
                    solver: LogisticRegressionSolverName::LBFGS,
                    alpha: 1e-6,
                },
            )
            .unwrap();
            let matrice_test = DenseMatrix::from_2d_vec(&x_test).unwrap();
            acc_lr[k] += precision(&y_test, &modele.predict(&matrice_test).unwrap());
        }
    }
    ns.iter()
        .enumerate()
        .map(|(k, _)| (acc_nb[k] / n_graines as f64, acc_lr[k] / n_graines as f64))
        .collect()
}

// matplotlib n'existe pas ici : la courbe est rendue en caracteres, deux colonnes
// par budget (o = Bayes a gauche, x = logistique a droite, * = collision).
fn courbe_ascii(ns: &[usize], nb: &[f64], lr: &[f64]) -> String {
    let lignes = 12usize;
    let min = nb.iter().chain(lr.iter()).cloned().fold(f64::INFINITY, f64::min);
    let max = nb.iter().chain(lr.iter()).cloned().fold(f64::NEG_INFINITY, f64::max);
    let largeur = ns.len() * 2;
    let mut grille = vec![vec![' '; largeur]; lignes];
    let ligne_de = |v: f64| -> usize {
        let t = (v - min) / (max - min + 1e-12);
        (((1.0 - t) * (lignes as f64 - 1.0)).round() as usize).min(lignes - 1)
    };
    for (k, (a, b)) in nb.iter().zip(lr.iter()).enumerate() {
        let la = ligne_de(*a);
        let lb = ligne_de(*b);
        if la == lb {
            grille[la][2 * k] = '*';
        } else {
            grille[la][2 * k] = 'o';
            grille[lb][2 * k + 1] = 'x';
        }
    }
    let mut sortie = String::new();
    for (i, ligne) in grille.iter().enumerate() {
        let valeur = max - (max - min) * (i as f64) / (lignes as f64 - 1.0);
        sortie.push_str(&format!("{:>7.3} | {}\n", valeur, ligne.iter().collect::<String>()));
    }
    sortie.push_str(&format!("        + {}\n", "-".repeat(largeur)));
    sortie.push_str(&format!(
        "          {}\n",
        ns.iter().map(|n| format!("{:<4}", n)).collect::<String>()
    ));
    sortie.push_str("          o = Bayes (colonne de gauche), x = logistique (colonne de droite), * = collision\n");
    sortie
}

let debut = Instant::now();
let ns = vec![15usize, 30, 60, 120, 240, 480, 960, 1920];
let resultats = eval_curve(make_corr, 12, 0.9, 0.6, &ns, 40, 1);
let (nb, lr): (Vec<f64>, Vec<f64>) = resultats.iter().cloned().unzip();

println!("== Scenario A : correlations homogenes, rho = 0.6, 12 dimensions ==");
println!("{}", courbe_ascii(&ns, &nb, &lr));
println!("{:>6} {:>8} {:>8} {:>8}", "n", "NB", "LR", "NB-LR");
for (k, &n) in ns.iter().enumerate() {
    println!("{:>6} {:>8.4} {:>8.4} {:>+8.4}", n, nb[k], lr[k], nb[k] - lr[k]);
}
println!("duree du scenario A = {:.1} s", debut.elapsed().as_secs_f64());

== Scenario A : correlations homogenes, rho = 0.6, 12 dimensions ==


  0.837 |              x x


  0.828 |     o o o * o o 


  0.820 |          x      


  0.812 |   o             


  0.803 |        x        


... (16 outputs left out)

   480   0.8297   0.8306  -0.0009


   960   0.8306   0.8347  -0.0041


  1920   0.8308   0.8367  -0.0059


duree du scenario A = 2.7 s


### Lecture du scénario A

Le renversement est net, et il tient en une phrase : **Naive Bayes gagne tant que l'échantillon est petit, la régression logistique le rattrape quand il grandit.**

- À $n = 15$, la régression logistique doit estimer treize coefficients avec quinze points : elle apprend surtout du bruit. Naive Bayes estime deux paramètres par dimension et par classe, mais son hypothèse gaussienne est *vraie* ici sur chaque dimension prise seule — elle encaisse le manque de données.
- L'écart culmine vers $n = 30$ (+0,068), puis fond régulièrement. Vers $n = 480$, les deux courbes se croisent ; au-delà, la régression logistique passe devant, de quatre dixièmes de point à $n = 960$ puis de six dixièmes à $n = 1920$.
- Ce croisement est **la prédiction asymptotique** : à grande taille, le modèle le mieux spécifié gagne. La régression logistique a treize paramètres libres, mais rien ne la contraint à la mauvaise direction ; Naive Bayes en estime quarante-huit et ne peut pas corriger son biais — il plafonne.

C'est le **compromis biais-variance** rendu visible, avec un seul levier : $n$. Le carnet Python, sur son propre générateur, ne montre qu'un resserrement de l'écart sans croisement ; l'ordre des deux courbes à grand $n$ dépend du terrain exact, et le registre de conformité ne mesure donc pas le signe à $n = 1920$ pour ce scénario.

In [8]:
use std::time::Instant;
use smartcore::linalg::basic::matrix::DenseMatrix;
use smartcore::linear::logistic_regression::{
    LogisticRegression, LogisticRegressionParameters, LogisticRegressionSolverName,
};

// Generateur maison : xorshift64 pour les entiers, Box-Muller pour le normal.
// Aucune crate `rand` : le flux est reproductible par graine, et identique d'une
// cellule a l'autre sans dependre d'une version tierce.
struct XorShift64 {
    etat: u64,
}

impl XorShift64 {
    fn new(graine: u64) -> XorShift64 {
        XorShift64 {
            etat: if graine == 0 { 0x9E3779B97F4A7C15 } else { graine },
        }
    }

    fn suivant(&mut self) -> u64 {
        let mut x = self.etat;
        x ^= x << 13;
        x ^= x >> 7;
        x ^= x << 17;
        self.etat = x;
        x
    }

    // Uniforme sur [0, 1) : 53 bits de mantisse, comme numpy.
    fn uniforme(&mut self) -> f64 {
        (self.suivant() >> 11) as f64 / ((1u64 << 53) as f64)
    }

    // Normale centree reduite par Box-Muller.
    fn normal(&mut self) -> f64 {
        let u1 = self.uniforme().max(1e-12);
        let u2 = self.uniforme();
        (-2.0 * u1.ln()).sqrt() * (2.0 * std::f64::consts::PI * u2).cos()
    }

    fn entier(&mut self, borne: u64) -> u64 {
        self.suivant() % borne
    }
}

fn sigmoide(z: f64) -> f64 {
    1.0 / (1.0 + (-z).exp())
}

fn precision(y_vrai: &[u32], y_pred: &[u32]) -> f64 {
    let bons = y_vrai
        .iter()
        .zip(y_pred.iter())
        .filter(|(a, b)| a == b)
        .count();
    bons as f64 / y_vrai.len() as f64
}

// Naive Bayes gaussien ecrit a la main.
// scipy.stats.norm.logpdf n'existe pas ici : la log-densite s'ecrit
//   log N(x ; mu, var) = -0.5 * ( log(2*pi*var) + (x - mu)^2 / var ).
struct MonGaussianNB {
    classes: Vec<u32>,
    moyens: Vec<Vec<f64>>,
    variances: Vec<Vec<f64>>,
    priors: Vec<f64>,
}

impl MonGaussianNB {
    fn fit(x: &[Vec<f64>], y: &[u32]) -> MonGaussianNB {
        let mut classes = y.to_vec();
        classes.sort_unstable();
        classes.dedup();
        let d = x[0].len();
        let n = x.len() as f64;
        let mut moyens = Vec::new();
        let mut variances = Vec::new();
        let mut priors = Vec::new();
        for &c in &classes {
            let lignes: Vec<&Vec<f64>> = x
                .iter()
                .zip(y.iter())
                .filter(|(_, yi)| **yi == c)
                .map(|(xi, _)| xi)
                .collect();
            let m = lignes.len() as f64;
            let mut mu = vec![0.0; d];
            for ligne in &lignes {
                for j in 0..d {
                    mu[j] += ligne[j];
                }
            }
            for j in 0..d {
                mu[j] /= m;
            }
            // Variance de population, puis l'epsilon de garde : sans lui une
            // dimension constante annule la densite (division par zero).
            let mut var = vec![0.0; d];
            for ligne in &lignes {
                for j in 0..d {
                    var[j] += (ligne[j] - mu[j]).powi(2);
                }
            }
            for j in 0..d {
                var[j] = var[j] / m + 1e-9;
            }
            moyens.push(mu);
            variances.push(var);
            priors.push(m / n);
        }
        MonGaussianNB { classes, moyens, variances, priors }
    }

    // Un score par classe et par ligne : log(prior) + somme des log-densites.
    fn log_vraisemblance(&self, x: &[Vec<f64>]) -> Vec<Vec<f64>> {
        x.iter()
            .map(|ligne| {
                self.classes
                    .iter()
                    .enumerate()
                    .map(|(k, _)| {
                        let mut score = self.priors[k].ln();
                        for j in 0..ligne.len() {
                            let var = self.variances[k][j];
                            let ecart = ligne[j] - self.moyens[k][j];
                            score += -0.5
                                * ((2.0 * std::f64::consts::PI * var).ln() + ecart * ecart / var);
                        }
                        score
                    })
                    .collect()
            })
            .collect()
    }

    fn predict(&self, x: &[Vec<f64>]) -> Vec<u32> {
        self.log_vraisemblance(x)
            .iter()
            .map(|scores| {
                let mut meilleur = 0usize;
                for k in 1..scores.len() {
                    if scores[k] > scores[meilleur] {
                        meilleur = k;
                    }
                }
                self.classes[meilleur]
            })
            .collect()
    }
}

// Scenario A -- correlations homogenes : un seul facteur latent porte le signal,
// toutes les dimensions en derivent avec le meme rho. C'est le terrain ou
// l'hypothese d'independance de Naive Bayes est fausse mais peu couteuse.
fn make_corr(n: usize, d: usize, decalage: f64, rho: f64, rng: &mut XorShift64) -> (Vec<Vec<f64>>, Vec<u32>) {
    let mut x = Vec::with_capacity(n);
    let mut y = Vec::with_capacity(n);
    for _ in 0..n {
        let facteur = rng.normal();
        let mut ligne = Vec::with_capacity(d);
        for _ in 0..d {
            let e = rng.normal();
            ligne.push(rho.sqrt() * facteur + (1.0 - rho).sqrt() * e);
        }
        let score: f64 = ligne.iter().take(d / 2).sum::<f64>() / ((d / 2) as f64).sqrt();
        let classe = if rng.uniforme() < sigmoide(score) { 1u32 } else { 0u32 };
        if classe == 1 {
            let poussee = decalage / (d as f64).sqrt();
            for v in ligne.iter_mut() {
                *v += poussee;
            }
        }
        x.push(ligne);
        y.push(classe);
    }
    (x, y)
}

// Scenario B -- correlations en bloc, et deux pieges pour l'hypothese
// d'independance. Les deux moities de dimensions suivent chacune son facteur
// latent, et le signal est la DIFFERENCE des deux ; surtout, l'echelle des
// dimensions varie d'un facteur huit a l'interieur de chaque bloc, alors que le
// decalage de classe est le meme en valeur absolue sur toutes. Naive Bayes lit
// chaque dimension separement et pondere donc de travers ; la regression
// logistique, elle, peut apprendre la direction jointe.
fn make_block(n: usize, d: usize, decalage: f64, rho: f64, rng: &mut XorShift64) -> (Vec<Vec<f64>>, Vec<u32>) {
    let mut x = Vec::with_capacity(n);
    let mut y = Vec::with_capacity(n);
    let moitie = d / 2;
    for _ in 0..n {
        let facteur_a = rng.normal();
        let facteur_b = rng.normal();
        let mut ligne = Vec::with_capacity(d);
        for j in 0..d {
            let e = rng.normal();
            let (facteur, k) = if j < moitie { (facteur_a, j) } else { (facteur_b, j - moitie) };
            let echelle = 2f64.powi(3 - (k as i32));
            ligne.push(echelle * (rho.sqrt() * facteur + (1.0 - rho).sqrt() * e));
        }
        let score = 4.0 * decalage * (facteur_a - facteur_b) / std::f64::consts::SQRT_2;
        let classe = if rng.uniforme() < sigmoide(score) { 1u32 } else { 0u32 };
        if classe == 1 {
            let poussee = decalage / (d as f64).sqrt();
            for v in ligne.iter_mut() {
                *v += poussee;
            }
        }
        x.push(ligne);
        y.push(classe);
    }
    (x, y)
}

// Protocole apparie : pour chaque graine, un seul tirage de taille n_max, dont
// les n premieres lignes servent d'entrainement et le reste de test. Le jeu de
// test est donc disjoint et de taille 2*n_max - n au minimum.
fn eval_curve(
    make: fn(usize, usize, f64, f64, &mut XorShift64) -> (Vec<Vec<f64>>, Vec<u32>),
    d: usize,
    decalage: f64,
    rho: f64,
    ns: &[usize],
    n_graines: usize,
    graine: u64,
) -> Vec<(f64, f64)> {
    let n_max = ns.iter().cloned().max().unwrap() * 2;
    let mut acc_nb = vec![0.0; ns.len()];
    let mut acc_lr = vec![0.0; ns.len()];
    for g in 0..n_graines {
        let mut rng = XorShift64::new(graine * 1000 + g as u64 + 1);
        let (x, y) = make(n_max, d, decalage, rho, &mut rng);
        for (k, &n) in ns.iter().enumerate() {
            let x_train = x[..n].to_vec();
            let y_train = y[..n].to_vec();
            let x_test = x[n..].to_vec();
            let y_test = y[n..].to_vec();

            let nb = MonGaussianNB::fit(&x_train, &y_train);
            acc_nb[k] += precision(&y_test, &nb.predict(&x_test));

            // alpha = 1e-6 joue le role du C = 1e6 du jumeau Python : la
            // regularisation est rendue negligeable devant la vraisemblance.
            let matrice = DenseMatrix::from_2d_vec(&x_train).unwrap();
            let modele = LogisticRegression::fit(
                &matrice,
                &y_train,
                LogisticRegressionParameters {
                    solver: LogisticRegressionSolverName::LBFGS,
                    alpha: 1e-6,
                },
            )
            .unwrap();
            let matrice_test = DenseMatrix::from_2d_vec(&x_test).unwrap();
            acc_lr[k] += precision(&y_test, &modele.predict(&matrice_test).unwrap());
        }
    }
    ns.iter()
        .enumerate()
        .map(|(k, _)| (acc_nb[k] / n_graines as f64, acc_lr[k] / n_graines as f64))
        .collect()
}

// matplotlib n'existe pas ici : la courbe est rendue en caracteres, deux colonnes
// par budget (o = Bayes a gauche, x = logistique a droite, * = collision).
fn courbe_ascii(ns: &[usize], nb: &[f64], lr: &[f64]) -> String {
    let lignes = 12usize;
    let min = nb.iter().chain(lr.iter()).cloned().fold(f64::INFINITY, f64::min);
    let max = nb.iter().chain(lr.iter()).cloned().fold(f64::NEG_INFINITY, f64::max);
    let largeur = ns.len() * 2;
    let mut grille = vec![vec![' '; largeur]; lignes];
    let ligne_de = |v: f64| -> usize {
        let t = (v - min) / (max - min + 1e-12);
        (((1.0 - t) * (lignes as f64 - 1.0)).round() as usize).min(lignes - 1)
    };
    for (k, (a, b)) in nb.iter().zip(lr.iter()).enumerate() {
        let la = ligne_de(*a);
        let lb = ligne_de(*b);
        if la == lb {
            grille[la][2 * k] = '*';
        } else {
            grille[la][2 * k] = 'o';
            grille[lb][2 * k + 1] = 'x';
        }
    }
    let mut sortie = String::new();
    for (i, ligne) in grille.iter().enumerate() {
        let valeur = max - (max - min) * (i as f64) / (lignes as f64 - 1.0);
        sortie.push_str(&format!("{:>7.3} | {}\n", valeur, ligne.iter().collect::<String>()));
    }
    sortie.push_str(&format!("        + {}\n", "-".repeat(largeur)));
    sortie.push_str(&format!(
        "          {}\n",
        ns.iter().map(|n| format!("{:<4}", n)).collect::<String>()
    ));
    sortie.push_str("          o = Bayes (colonne de gauche), x = logistique (colonne de droite), * = collision\n");
    sortie
}

let debut = Instant::now();
let ns = vec![15usize, 30, 60, 120, 240, 480, 960, 1920];
let resultats = eval_curve(make_block, 12, 0.7, 0.85, &ns, 40, 2);
let (nb, lr): (Vec<f64>, Vec<f64>) = resultats.iter().cloned().unzip();

println!("== Scenario B : correlations en bloc, rho = 0.85, 12 dimensions ==");
println!("{}", courbe_ascii(&ns, &nb, &lr));
println!("{:>6} {:>8} {:>8} {:>8}", "n", "NB", "LR", "NB-LR");
for (k, &n) in ns.iter().enumerate() {
    println!("{:>6} {:>8.4} {:>8.4} {:>+8.4}", n, nb[k], lr[k], nb[k] - lr[k]);
}
println!("duree du scenario B = {:.1} s", debut.elapsed().as_secs_f64());

== Scenario B : correlations en bloc, rho = 0.85, 12 dimensions ==


  0.937 |            x x x


  0.919 |          x      


  0.900 |        x        


  0.882 |                 


  0.864 |      x          


... (16 outputs left out)

   480   0.8419   0.9335  -0.0915


   960   0.8421   0.9351  -0.0930


  1920   0.8423   0.9370  -0.0947


duree du scenario B = 22.7 s


### Lecture du scénario B

Le résultat s'inverse **et l'écart ne se referme pas**. Dès que l'échantillon quitte le tout petit régime, la régression logistique s'envole et laisse Naive Bayes loin derrière — de l'ordre de neuf points de précision à partir de $n = 240$, et qui ne diminuent plus.

Ce qui a changé n'est pas la quantité de données, c'est la **forme** de l'hypothèse fausse.

- En A, l'indépendance était fausse mais *symétrique* : toutes les dimensions portent la même information et se trompent de la même façon. Naive Bayes se trompe sur sa confiance, pas sur sa direction de décision.
- En B, les dimensions d'un bloc sont presque des copies les unes des autres (corrélation 0,85) et **leurs échelles diffèrent d'un facteur huit**, alors que le décalage de classe est le même en valeur absolue partout. Naive Bayes pondère donc chaque dimension par $1/\sigma_j^2$ : il donne un poids énorme aux dimensions à petite échelle et néglige les autres. La direction qu'il apprend n'est plus celle de l'optimum — et aucune quantité de données ne corrige un biais de spécification.
- La régression logistique n'a aucune de ces hypothèses : elle estime les douze coefficients joints, et l'échelle des dimensions lui est indifférente tant que la frontière reste linéaire (ici elle l'est).

**La conclusion n'est donc pas « le discriminatif est meilleur ».** En A, Naive Bayes gagne à petit $n$ ; en B, il ne gagne jamais au-delà du bruit d'échantillonnage. C'est la **structure du terrain** qui décide, et le seul moyen de le savoir est de mesurer — ce que le carnet Python et ce carnet font chacun sur son propre générateur, avec le même verdict de régime.

## Exercice 2 — la dimension change-t-elle le verdict ?

Les deux scénarios précédents vivent en 12 dimensions. Or le biais de Naive Bayes en B vient d'un **désalignement entre la direction optimale et les variances marginales** : cette quantité dépend de la dimension.

Complétez `ecarts_dimension` : pour chaque dimension demandée, relancez la courbe du scénario A et renvoyez `(dimension, écart NB − LR)` au budget $n = 30$.

Les deux jeux de test sont `([4, 12, 24], graine 1)` et `([4], graine 3)` — un balayage de la dimension et un seul point, pour vérifier que la fonction ne dépend pas d'un effet de bord de boucle. Le carnet compare votre écart à celui qu'il recalcule : la tolérance est de $10^{-9}$, ces calculs étant déterministes à graine fixée.

In [9]:
use std::time::Instant;
use smartcore::linalg::basic::matrix::DenseMatrix;
use smartcore::linear::logistic_regression::{
    LogisticRegression, LogisticRegressionParameters, LogisticRegressionSolverName,
};

// Generateur maison : xorshift64 pour les entiers, Box-Muller pour le normal.
// Aucune crate `rand` : le flux est reproductible par graine, et identique d'une
// cellule a l'autre sans dependre d'une version tierce.
struct XorShift64 {
    etat: u64,
}

impl XorShift64 {
    fn new(graine: u64) -> XorShift64 {
        XorShift64 {
            etat: if graine == 0 { 0x9E3779B97F4A7C15 } else { graine },
        }
    }

    fn suivant(&mut self) -> u64 {
        let mut x = self.etat;
        x ^= x << 13;
        x ^= x >> 7;
        x ^= x << 17;
        self.etat = x;
        x
    }

    // Uniforme sur [0, 1) : 53 bits de mantisse, comme numpy.
    fn uniforme(&mut self) -> f64 {
        (self.suivant() >> 11) as f64 / ((1u64 << 53) as f64)
    }

    // Normale centree reduite par Box-Muller.
    fn normal(&mut self) -> f64 {
        let u1 = self.uniforme().max(1e-12);
        let u2 = self.uniforme();
        (-2.0 * u1.ln()).sqrt() * (2.0 * std::f64::consts::PI * u2).cos()
    }

    fn entier(&mut self, borne: u64) -> u64 {
        self.suivant() % borne
    }
}

fn sigmoide(z: f64) -> f64 {
    1.0 / (1.0 + (-z).exp())
}

fn precision(y_vrai: &[u32], y_pred: &[u32]) -> f64 {
    let bons = y_vrai
        .iter()
        .zip(y_pred.iter())
        .filter(|(a, b)| a == b)
        .count();
    bons as f64 / y_vrai.len() as f64
}

// Naive Bayes gaussien ecrit a la main.
// scipy.stats.norm.logpdf n'existe pas ici : la log-densite s'ecrit
//   log N(x ; mu, var) = -0.5 * ( log(2*pi*var) + (x - mu)^2 / var ).
struct MonGaussianNB {
    classes: Vec<u32>,
    moyens: Vec<Vec<f64>>,
    variances: Vec<Vec<f64>>,
    priors: Vec<f64>,
}

impl MonGaussianNB {
    fn fit(x: &[Vec<f64>], y: &[u32]) -> MonGaussianNB {
        let mut classes = y.to_vec();
        classes.sort_unstable();
        classes.dedup();
        let d = x[0].len();
        let n = x.len() as f64;
        let mut moyens = Vec::new();
        let mut variances = Vec::new();
        let mut priors = Vec::new();
        for &c in &classes {
            let lignes: Vec<&Vec<f64>> = x
                .iter()
                .zip(y.iter())
                .filter(|(_, yi)| **yi == c)
                .map(|(xi, _)| xi)
                .collect();
            let m = lignes.len() as f64;
            let mut mu = vec![0.0; d];
            for ligne in &lignes {
                for j in 0..d {
                    mu[j] += ligne[j];
                }
            }
            for j in 0..d {
                mu[j] /= m;
            }
            // Variance de population, puis l'epsilon de garde : sans lui une
            // dimension constante annule la densite (division par zero).
            let mut var = vec![0.0; d];
            for ligne in &lignes {
                for j in 0..d {
                    var[j] += (ligne[j] - mu[j]).powi(2);
                }
            }
            for j in 0..d {
                var[j] = var[j] / m + 1e-9;
            }
            moyens.push(mu);
            variances.push(var);
            priors.push(m / n);
        }
        MonGaussianNB { classes, moyens, variances, priors }
    }

    // Un score par classe et par ligne : log(prior) + somme des log-densites.
    fn log_vraisemblance(&self, x: &[Vec<f64>]) -> Vec<Vec<f64>> {
        x.iter()
            .map(|ligne| {
                self.classes
                    .iter()
                    .enumerate()
                    .map(|(k, _)| {
                        let mut score = self.priors[k].ln();
                        for j in 0..ligne.len() {
                            let var = self.variances[k][j];
                            let ecart = ligne[j] - self.moyens[k][j];
                            score += -0.5
                                * ((2.0 * std::f64::consts::PI * var).ln() + ecart * ecart / var);
                        }
                        score
                    })
                    .collect()
            })
            .collect()
    }

    fn predict(&self, x: &[Vec<f64>]) -> Vec<u32> {
        self.log_vraisemblance(x)
            .iter()
            .map(|scores| {
                let mut meilleur = 0usize;
                for k in 1..scores.len() {
                    if scores[k] > scores[meilleur] {
                        meilleur = k;
                    }
                }
                self.classes[meilleur]
            })
            .collect()
    }
}

// Scenario A -- correlations homogenes : un seul facteur latent porte le signal,
// toutes les dimensions en derivent avec le meme rho. C'est le terrain ou
// l'hypothese d'independance de Naive Bayes est fausse mais peu couteuse.
fn make_corr(n: usize, d: usize, decalage: f64, rho: f64, rng: &mut XorShift64) -> (Vec<Vec<f64>>, Vec<u32>) {
    let mut x = Vec::with_capacity(n);
    let mut y = Vec::with_capacity(n);
    for _ in 0..n {
        let facteur = rng.normal();
        let mut ligne = Vec::with_capacity(d);
        for _ in 0..d {
            let e = rng.normal();
            ligne.push(rho.sqrt() * facteur + (1.0 - rho).sqrt() * e);
        }
        let score: f64 = ligne.iter().take(d / 2).sum::<f64>() / ((d / 2) as f64).sqrt();
        let classe = if rng.uniforme() < sigmoide(score) { 1u32 } else { 0u32 };
        if classe == 1 {
            let poussee = decalage / (d as f64).sqrt();
            for v in ligne.iter_mut() {
                *v += poussee;
            }
        }
        x.push(ligne);
        y.push(classe);
    }
    (x, y)
}

// Scenario B -- correlations en bloc, et deux pieges pour l'hypothese
// d'independance. Les deux moities de dimensions suivent chacune son facteur
// latent, et le signal est la DIFFERENCE des deux ; surtout, l'echelle des
// dimensions varie d'un facteur huit a l'interieur de chaque bloc, alors que le
// decalage de classe est le meme en valeur absolue sur toutes. Naive Bayes lit
// chaque dimension separement et pondere donc de travers ; la regression
// logistique, elle, peut apprendre la direction jointe.
fn make_block(n: usize, d: usize, decalage: f64, rho: f64, rng: &mut XorShift64) -> (Vec<Vec<f64>>, Vec<u32>) {
    let mut x = Vec::with_capacity(n);
    let mut y = Vec::with_capacity(n);
    let moitie = d / 2;
    for _ in 0..n {
        let facteur_a = rng.normal();
        let facteur_b = rng.normal();
        let mut ligne = Vec::with_capacity(d);
        for j in 0..d {
            let e = rng.normal();
            let (facteur, k) = if j < moitie { (facteur_a, j) } else { (facteur_b, j - moitie) };
            let echelle = 2f64.powi(3 - (k as i32));
            ligne.push(echelle * (rho.sqrt() * facteur + (1.0 - rho).sqrt() * e));
        }
        let score = 4.0 * decalage * (facteur_a - facteur_b) / std::f64::consts::SQRT_2;
        let classe = if rng.uniforme() < sigmoide(score) { 1u32 } else { 0u32 };
        if classe == 1 {
            let poussee = decalage / (d as f64).sqrt();
            for v in ligne.iter_mut() {
                *v += poussee;
            }
        }
        x.push(ligne);
        y.push(classe);
    }
    (x, y)
}

// Protocole apparie : pour chaque graine, un seul tirage de taille n_max, dont
// les n premieres lignes servent d'entrainement et le reste de test. Le jeu de
// test est donc disjoint et de taille 2*n_max - n au minimum.
fn eval_curve(
    make: fn(usize, usize, f64, f64, &mut XorShift64) -> (Vec<Vec<f64>>, Vec<u32>),
    d: usize,
    decalage: f64,
    rho: f64,
    ns: &[usize],
    n_graines: usize,
    graine: u64,
) -> Vec<(f64, f64)> {
    let n_max = ns.iter().cloned().max().unwrap() * 2;
    let mut acc_nb = vec![0.0; ns.len()];
    let mut acc_lr = vec![0.0; ns.len()];
    for g in 0..n_graines {
        let mut rng = XorShift64::new(graine * 1000 + g as u64 + 1);
        let (x, y) = make(n_max, d, decalage, rho, &mut rng);
        for (k, &n) in ns.iter().enumerate() {
            let x_train = x[..n].to_vec();
            let y_train = y[..n].to_vec();
            let x_test = x[n..].to_vec();
            let y_test = y[n..].to_vec();

            let nb = MonGaussianNB::fit(&x_train, &y_train);
            acc_nb[k] += precision(&y_test, &nb.predict(&x_test));

            // alpha = 1e-6 joue le role du C = 1e6 du jumeau Python : la
            // regularisation est rendue negligeable devant la vraisemblance.
            let matrice = DenseMatrix::from_2d_vec(&x_train).unwrap();
            let modele = LogisticRegression::fit(
                &matrice,
                &y_train,
                LogisticRegressionParameters {
                    solver: LogisticRegressionSolverName::LBFGS,
                    alpha: 1e-6,
                },
            )
            .unwrap();
            let matrice_test = DenseMatrix::from_2d_vec(&x_test).unwrap();
            acc_lr[k] += precision(&y_test, &modele.predict(&matrice_test).unwrap());
        }
    }
    ns.iter()
        .enumerate()
        .map(|(k, _)| (acc_nb[k] / n_graines as f64, acc_lr[k] / n_graines as f64))
        .collect()
}

// Exercice 2 : ecart de precision NB - LR au budget n = 30, pour chaque dimension.
fn ecarts_dimension(
    ns: &[usize],
    dimensions: &[usize],
    graine: u64,
) -> Option<Vec<(usize, f64)>> {
    // TODO : pour chaque dimension d, appeler eval_curve(make_corr, d, 0.9, 0.6, ns, 40, graine)
    // et renvoyer (d, NB(n = 30) - LR(n = 30)). L'indice du budget 30 dans `ns`
    // est ns.iter().position(|&x| x == 30).
    None
}

let ns = vec![15usize, 30, 60, 120, 240, 480, 960, 1920];
let fixtures: Vec<(Vec<usize>, u64)> = vec![(vec![4, 12, 24], 1), (vec![4], 3)];
let reponses: Vec<Option<Vec<(usize, f64)>>> = fixtures
    .iter()
    .map(|(dims, g)| ecarts_dimension(&ns, dims, *g))
    .collect();

if reponses.iter().all(|r| r.is_none()) {
    println!("Exercice 2 à compléter.");
} else if reponses.iter().any(|r| r.is_none()) {
    println!("Exercice 2 : à revoir");
} else {
    let mut conforme = true;
    for ((dims, g), reponse) in fixtures.iter().zip(reponses.iter()) {
        let resultat = reponse.as_ref().unwrap();
        conforme = conforme && resultat.len() == dims.len();
        if !conforme {
            break;
        }
        let position = ns.iter().position(|&x| x == 30).unwrap();
        for (k, &dim) in dims.iter().enumerate() {
            let courbe = eval_curve(make_corr, dim, 0.9, 0.6, &ns, 40, *g);
            let attendu = courbe[position].0 - courbe[position].1;
            conforme = conforme && resultat[k].0 == dim && (resultat[k].1 - attendu).abs() < 1e-9;
        }
    }
    if conforme {
        println!("Exercice 2 : écarts conformes ; lire la tendance selon la dimension");
    } else {
        println!("Exercice 2 : à revoir");
    }
};

Exercice 2 à compléter.


### Guide de l'exercice 2

La structure est courte, mais deux détails comptent.

1. **Retrouver l'indice du budget.** `ns.iter().position(|&x| x == 30)` rend l'indice de $n = 30$ dans le vecteur des budgets — c'est la ligne `courbe[position]` qui porte l'écart cherché. Coder `courbe[1]` en dur fonctionne ici et casse au premier changement de budgets : la fonction prend `ns` en argument, elle doit le lire.
2. **Passer `ns` tel quel.** `eval_curve` renvoie un couple par budget, dans l'ordre de `ns`. Ne pas re-trier, ne pas tronquer : la fonction est déterministe à graine fixée, donc le carnet peut recalculer la même courbe et comparer à $10^{-9}$.

Sur le fond, attendez-vous à une tendance, pas à une loi : à $n = 30$ le régime est encore dominé par la variance d'estimation de la régression logistique, et l'écart en faveur de Naive Bayes se resserre à mesure que la dimension augmente, parce que le nombre de coefficients à estimer croît avec elle.

## 3. Quand choisir laquelle ?

| Situation | Voie générative (Naive Bayes) | Voie discriminative (logistique) |
|---|---|---|
| Très peu de données par classe ($n$ de l'ordre de la dizaine) | **Avantage** : peu de paramètres, convergence immédiate | Treize coefficients à estimer avec quinze points : bruit |
| Beaucoup de dimensions, peu d'exemples | **Avantage** : coût linéaire en $d$, pas de matrice à inverser | Sur-apprentissage, sauf régularisation forte |
| Hypothèse d'indépendance fausse de façon **uniforme** | Reste compétitif : le biais porte sur la confiance, pas sur la direction | Converge vers l'optimum à grand $n$ |
| Hypothèse fausse de façon **structurée** (blocs, échelles hétérogènes) | **Perd, et l'écart ne se referme pas** — c'est le scénario B | **Avantage net** : apprend la direction jointe |
| Frontière non linéaire | Aucun des deux ne la voit | Aucun des deux ne la voit (il faut noyau, arbres, réseau) |
| Coût de calcul | Un passage sur les données : $O(nd)$ | Itérations d'un solveur (ici LBFGS) |
| Interprétation | Moyennes et variances par classe, très lisibles | Coefficients, avec un signe par dimension |

La ligne la plus utile est la quatrième : **ce n'est pas la quantité de données qui départage les deux voies, c'est la structure de ce que l'hypothèse d'indépendance rate.** En A, l'hypothèse est fausse et cela coûte peu ; en B, elle est fausse et cela coûte neuf points de précision, définitivement.

Un mot sur ce que le tableau ne dit pas : il ne dit rien du **calibrage des probabilités**. Naive Bayes, avec des dimensions corrélées, produit des probabilités trop tranchées (proches de 0 et 1) même quand son argmax est bon. Si la décision utilise la probabilité elle-même — un coût asymétrique, un seuil métier —, ce biais de calibrage compte autant que la précision, et il n'est mesuré nulle part dans ce carnet : c'est un exercice pour le lecteur.

## Exercice 3 — lire les deux régimes ensemble

Les deux tables précédentes disent *qui gagne*. Elles ne disent pas *pourquoi*, et c'est là que se joue la compréhension.

Complétez `analyse_regimes` : elle reçoit les deux courbes (A puis B), le vecteur des budgets et un budget $n$, et renvoie `(écart A, écart B, explication)` au budget demandé — les écarts étant NB − LR, et l'explication une phrase d'au moins vingt caractères qui nomme le mécanisme.

Le carnet vérifie les deux écarts à $10^{-9}$ et exige une explication non vide pour les deux budgets testés, $n = 30$ et $n = 120$. Le contenu de la phrase n'est pas jugé par un motif : c'est vous qui le relisez, à la lumière des deux lectures ci-dessus.

In [10]:
use std::time::Instant;
use smartcore::linalg::basic::matrix::DenseMatrix;
use smartcore::linear::logistic_regression::{
    LogisticRegression, LogisticRegressionParameters, LogisticRegressionSolverName,
};

// Generateur maison : xorshift64 pour les entiers, Box-Muller pour le normal.
// Aucune crate `rand` : le flux est reproductible par graine, et identique d'une
// cellule a l'autre sans dependre d'une version tierce.
struct XorShift64 {
    etat: u64,
}

impl XorShift64 {
    fn new(graine: u64) -> XorShift64 {
        XorShift64 {
            etat: if graine == 0 { 0x9E3779B97F4A7C15 } else { graine },
        }
    }

    fn suivant(&mut self) -> u64 {
        let mut x = self.etat;
        x ^= x << 13;
        x ^= x >> 7;
        x ^= x << 17;
        self.etat = x;
        x
    }

    // Uniforme sur [0, 1) : 53 bits de mantisse, comme numpy.
    fn uniforme(&mut self) -> f64 {
        (self.suivant() >> 11) as f64 / ((1u64 << 53) as f64)
    }

    // Normale centree reduite par Box-Muller.
    fn normal(&mut self) -> f64 {
        let u1 = self.uniforme().max(1e-12);
        let u2 = self.uniforme();
        (-2.0 * u1.ln()).sqrt() * (2.0 * std::f64::consts::PI * u2).cos()
    }

    fn entier(&mut self, borne: u64) -> u64 {
        self.suivant() % borne
    }
}

fn sigmoide(z: f64) -> f64 {
    1.0 / (1.0 + (-z).exp())
}

fn precision(y_vrai: &[u32], y_pred: &[u32]) -> f64 {
    let bons = y_vrai
        .iter()
        .zip(y_pred.iter())
        .filter(|(a, b)| a == b)
        .count();
    bons as f64 / y_vrai.len() as f64
}

// Naive Bayes gaussien ecrit a la main.
// scipy.stats.norm.logpdf n'existe pas ici : la log-densite s'ecrit
//   log N(x ; mu, var) = -0.5 * ( log(2*pi*var) + (x - mu)^2 / var ).
struct MonGaussianNB {
    classes: Vec<u32>,
    moyens: Vec<Vec<f64>>,
    variances: Vec<Vec<f64>>,
    priors: Vec<f64>,
}

impl MonGaussianNB {
    fn fit(x: &[Vec<f64>], y: &[u32]) -> MonGaussianNB {
        let mut classes = y.to_vec();
        classes.sort_unstable();
        classes.dedup();
        let d = x[0].len();
        let n = x.len() as f64;
        let mut moyens = Vec::new();
        let mut variances = Vec::new();
        let mut priors = Vec::new();
        for &c in &classes {
            let lignes: Vec<&Vec<f64>> = x
                .iter()
                .zip(y.iter())
                .filter(|(_, yi)| **yi == c)
                .map(|(xi, _)| xi)
                .collect();
            let m = lignes.len() as f64;
            let mut mu = vec![0.0; d];
            for ligne in &lignes {
                for j in 0..d {
                    mu[j] += ligne[j];
                }
            }
            for j in 0..d {
                mu[j] /= m;
            }
            // Variance de population, puis l'epsilon de garde : sans lui une
            // dimension constante annule la densite (division par zero).
            let mut var = vec![0.0; d];
            for ligne in &lignes {
                for j in 0..d {
                    var[j] += (ligne[j] - mu[j]).powi(2);
                }
            }
            for j in 0..d {
                var[j] = var[j] / m + 1e-9;
            }
            moyens.push(mu);
            variances.push(var);
            priors.push(m / n);
        }
        MonGaussianNB { classes, moyens, variances, priors }
    }

    // Un score par classe et par ligne : log(prior) + somme des log-densites.
    fn log_vraisemblance(&self, x: &[Vec<f64>]) -> Vec<Vec<f64>> {
        x.iter()
            .map(|ligne| {
                self.classes
                    .iter()
                    .enumerate()
                    .map(|(k, _)| {
                        let mut score = self.priors[k].ln();
                        for j in 0..ligne.len() {
                            let var = self.variances[k][j];
                            let ecart = ligne[j] - self.moyens[k][j];
                            score += -0.5
                                * ((2.0 * std::f64::consts::PI * var).ln() + ecart * ecart / var);
                        }
                        score
                    })
                    .collect()
            })
            .collect()
    }

    fn predict(&self, x: &[Vec<f64>]) -> Vec<u32> {
        self.log_vraisemblance(x)
            .iter()
            .map(|scores| {
                let mut meilleur = 0usize;
                for k in 1..scores.len() {
                    if scores[k] > scores[meilleur] {
                        meilleur = k;
                    }
                }
                self.classes[meilleur]
            })
            .collect()
    }
}

// Scenario A -- correlations homogenes : un seul facteur latent porte le signal,
// toutes les dimensions en derivent avec le meme rho. C'est le terrain ou
// l'hypothese d'independance de Naive Bayes est fausse mais peu couteuse.
fn make_corr(n: usize, d: usize, decalage: f64, rho: f64, rng: &mut XorShift64) -> (Vec<Vec<f64>>, Vec<u32>) {
    let mut x = Vec::with_capacity(n);
    let mut y = Vec::with_capacity(n);
    for _ in 0..n {
        let facteur = rng.normal();
        let mut ligne = Vec::with_capacity(d);
        for _ in 0..d {
            let e = rng.normal();
            ligne.push(rho.sqrt() * facteur + (1.0 - rho).sqrt() * e);
        }
        let score: f64 = ligne.iter().take(d / 2).sum::<f64>() / ((d / 2) as f64).sqrt();
        let classe = if rng.uniforme() < sigmoide(score) { 1u32 } else { 0u32 };
        if classe == 1 {
            let poussee = decalage / (d as f64).sqrt();
            for v in ligne.iter_mut() {
                *v += poussee;
            }
        }
        x.push(ligne);
        y.push(classe);
    }
    (x, y)
}

// Scenario B -- correlations en bloc, et deux pieges pour l'hypothese
// d'independance. Les deux moities de dimensions suivent chacune son facteur
// latent, et le signal est la DIFFERENCE des deux ; surtout, l'echelle des
// dimensions varie d'un facteur huit a l'interieur de chaque bloc, alors que le
// decalage de classe est le meme en valeur absolue sur toutes. Naive Bayes lit
// chaque dimension separement et pondere donc de travers ; la regression
// logistique, elle, peut apprendre la direction jointe.
fn make_block(n: usize, d: usize, decalage: f64, rho: f64, rng: &mut XorShift64) -> (Vec<Vec<f64>>, Vec<u32>) {
    let mut x = Vec::with_capacity(n);
    let mut y = Vec::with_capacity(n);
    let moitie = d / 2;
    for _ in 0..n {
        let facteur_a = rng.normal();
        let facteur_b = rng.normal();
        let mut ligne = Vec::with_capacity(d);
        for j in 0..d {
            let e = rng.normal();
            let (facteur, k) = if j < moitie { (facteur_a, j) } else { (facteur_b, j - moitie) };
            let echelle = 2f64.powi(3 - (k as i32));
            ligne.push(echelle * (rho.sqrt() * facteur + (1.0 - rho).sqrt() * e));
        }
        let score = 4.0 * decalage * (facteur_a - facteur_b) / std::f64::consts::SQRT_2;
        let classe = if rng.uniforme() < sigmoide(score) { 1u32 } else { 0u32 };
        if classe == 1 {
            let poussee = decalage / (d as f64).sqrt();
            for v in ligne.iter_mut() {
                *v += poussee;
            }
        }
        x.push(ligne);
        y.push(classe);
    }
    (x, y)
}

// Protocole apparie : pour chaque graine, un seul tirage de taille n_max, dont
// les n premieres lignes servent d'entrainement et le reste de test. Le jeu de
// test est donc disjoint et de taille 2*n_max - n au minimum.
fn eval_curve(
    make: fn(usize, usize, f64, f64, &mut XorShift64) -> (Vec<Vec<f64>>, Vec<u32>),
    d: usize,
    decalage: f64,
    rho: f64,
    ns: &[usize],
    n_graines: usize,
    graine: u64,
) -> Vec<(f64, f64)> {
    let n_max = ns.iter().cloned().max().unwrap() * 2;
    let mut acc_nb = vec![0.0; ns.len()];
    let mut acc_lr = vec![0.0; ns.len()];
    for g in 0..n_graines {
        let mut rng = XorShift64::new(graine * 1000 + g as u64 + 1);
        let (x, y) = make(n_max, d, decalage, rho, &mut rng);
        for (k, &n) in ns.iter().enumerate() {
            let x_train = x[..n].to_vec();
            let y_train = y[..n].to_vec();
            let x_test = x[n..].to_vec();
            let y_test = y[n..].to_vec();

            let nb = MonGaussianNB::fit(&x_train, &y_train);
            acc_nb[k] += precision(&y_test, &nb.predict(&x_test));

            // alpha = 1e-6 joue le role du C = 1e6 du jumeau Python : la
            // regularisation est rendue negligeable devant la vraisemblance.
            let matrice = DenseMatrix::from_2d_vec(&x_train).unwrap();
            let modele = LogisticRegression::fit(
                &matrice,
                &y_train,
                LogisticRegressionParameters {
                    solver: LogisticRegressionSolverName::LBFGS,
                    alpha: 1e-6,
                },
            )
            .unwrap();
            let matrice_test = DenseMatrix::from_2d_vec(&x_test).unwrap();
            acc_lr[k] += precision(&y_test, &modele.predict(&matrice_test).unwrap());
        }
    }
    ns.iter()
        .enumerate()
        .map(|(k, _)| (acc_nb[k] / n_graines as f64, acc_lr[k] / n_graines as f64))
        .collect()
}

// Exercice 3 : lire les deux regimes au meme budget.
fn analyse_regimes(
    res_a: &[(f64, f64)],
    res_b: &[(f64, f64)],
    ns: &[usize],
    n: usize,
) -> Option<(f64, f64, String)> {
    // TODO : retrouver l'indice du budget n, renvoyer les deux ecarts NB - LR et
    // une phrase qui nomme le mecanisme (au moins vingt caracteres).
    None
}

let ns = vec![15usize, 30, 60, 120, 240, 480, 960, 1920];
let res_a = eval_curve(make_corr, 12, 0.9, 0.6, &ns, 40, 1);
let res_b = eval_curve(make_block, 12, 0.7, 0.85, &ns, 40, 2);
let budgets_testes = [30usize, 120];
let reponses: Vec<Option<(f64, f64, String)>> = budgets_testes
    .iter()
    .map(|n| analyse_regimes(&res_a, &res_b, &ns, *n))
    .collect();

if reponses.iter().all(|r| r.is_none()) {
    println!("Exercice 3 à compléter.");
} else if reponses.iter().any(|r| r.is_none()) {
    println!("Exercice 3 : à revoir");
} else {
    let mut conforme = true;
    for (k, &n) in budgets_testes.iter().enumerate() {
        let (ecart_a, ecart_b, explication) = reponses[k].as_ref().unwrap();
        let position = ns.iter().position(|&x| x == n).unwrap();
        let attendu_a = res_a[position].0 - res_a[position].1;
        let attendu_b = res_b[position].0 - res_b[position].1;
        conforme = conforme
            && (ecart_a - attendu_a).abs() < 1e-9
            && (ecart_b - attendu_b).abs() < 1e-9
            && explication.trim().len() >= 20;
    }
    if conforme {
        println!("Exercice 3 : écarts conformes ; discuter le mécanisme de chaque régime");
    } else {
        println!("Exercice 3 : à revoir");
    }
};

Exercice 3 à compléter.


### Guide de l'exercice 3

Trois pièges, dans l'ordre où ils se présentent.

1. **Les deux courbes ont la même longueur, pas le même contenu.** `res_a[k]` et `res_b[k]` sont des couples `(NB, LR)` pour le budget `ns[k]` — l'écart est `res_a[k].0 - res_a[k].1`. Lire `.1 - .0` inverse le signe et le carnet le détecte.
2. **L'indice se retrouve, il ne se suppose pas.** `ns.iter().position(|&x| x == n)` protège contre un changement de budgets ; un `match n { 30 => 1, 120 => 3, _ => 0 }` passe l'exercice et ment au premier budget ajouté.
3. **L'explication est un livrable, pas une formalité.** La phrase attendue nomme le mécanisme, et il n'y en a que deux dans ce carnet : en A, l'hypothèse d'indépendance est fausse mais *uniforme* (biais de confiance, direction intacte) ; en B, elle est fausse de façon *structurée* (échelles hétérogènes, direction désalignée, biais de spécification que les données ne corrigent pas).

La question à laquelle votre phrase doit répondre : *pourquoi l'écart en B ne se referme-t-il pas quand $n$ grandit ?* Si votre réponse contient « plus de données » ou « surapprentissage », elle décrit autre chose — en A, c'est justement le manque de données qui fait perdre la logistique.

## À retenir

| Notion | Ce que le carnet a montré |
|---|---|
| **Naive Bayes gaussien** | $\log \pi_c + \sum_j \log \mathcal{N}(x_j; \mu_{jc}, \sigma^2_{jc})$ — un produit de densités pris en logarithme, un epsilon de garde sur la variance |
| **Vérification croisée** | L'implémentation écrite à la main rend les **mêmes prédictions** que `smartcore` sur les mêmes données : la vérification porte sur les prédictions, la bibliothèque n'exposant pas ses moyennes |
| **Définitions** | Le génératif modélise $P(x \mid y)$ puis Bayes ; le discriminatif modélise $P(y \mid x)$ directement |
| **Compromis biais-variance** | En A (corrélations homogènes), Naive Bayes gagne à petit $n$ et la logistique le rattrape : c'est le **biais** du génératif qui se paie, et la **variance** du discriminatif qui se paie d'abord |
| **Biais de spécification** | En B (corrélations en bloc, échelles hétérogènes), l'écart ne se referme **jamais** : aucune quantité de données ne corrige une hypothèse fausse de façon structurée |
| **Ce qui départage** | Ni la quantité de données seule, ni « le SOTA » : la **structure du terrain** — homogène ou en bloc |
| **Ce que le portage change** | Les niveaux absolus (deux générateurs) ; pas les régimes (le registre de conformité le vérifie) |

La leçon transférable : avant de choisir un classifieur, demandez-vous **de quelle façon votre hypothèse risque d'être fausse**. Uniformément — le génératif reste une bonne affaire, surtout avec peu de données. Structurellement — aucun raccourci ne tient, et il faut un modèle qui voit la covariance jointe.

## Questions fréquentes

<details>
<summary>Pourquoi Naive Bayes gagne-t-il alors que son hypothèse est évidemment fausse ?</summary>

Parce que la décision ne dépend pas de la qualité de l'estimation des densités, mais de la **direction** qu'elles induisent. En scénario A, toutes les dimensions sont corrélées au même facteur : l'hypothèse d'indépendance gonfle la confiance (le score croît $d$ fois plus vite qu'il ne devrait), mais l'argmax reste le bon, car la direction de décision ne change pas. Naive Bayes paie donc un biais de **calibrage**, pas un biais de **classement** — d'où sa robustesse en classification binaire à classes équilibrées, et sa fragilité dès qu'on lit ses probabilités comme des probabilités.

</details>

<details>
<summary>Pourquoi la logistique ne rattrape-t-elle pas son retard en scénario B ?</summary>

Parce que le retard de Naive Bayes en B n'est pas un retard de variance. À $n = 240$ il y a déjà près de 4 000 points de test : l'estimation est stable, et les courbes sont plates au-delà. Ce qui reste est un **biais de spécification** : la règle de Naive Bayes pondère chaque dimension par $1/\sigma_j^2$, alors que l'optimum dépend de la matrice de covariance jointe. Quand les échelles varient d'un facteur huit dans un bloc corrélé à 0,85, cette pondération marginale désigne une direction qui n'est pas celle de la marge optimale. Aucun supplément de données ne change une direction : il ne fait que mieux estimer la mauvaise.

</details>

<details>
<summary>Faut-il en conclure qu'il faut toujours préférer la régression logistique ?</summary>

Non, et c'est l'objet du scénario A. Avec quinze points par tirage, la logistique est systématiquement derrière — elle doit estimer treize coefficients là où Naive Bayes en estime vingt-cinq mais avec une structure qui les contraint. Sur des problèmes à beaucoup de dimensions et peu d'exemples (classification de texte, diagnostic rare), ce régime est la règle plutôt que l'exception, et c'est là que Naive Bayes reste un choix de premier rang. La bonne question n'est pas « quel modèle est meilleur » mais « quelle hypothèse mon terrain viole-t-il, et à quel prix ».

</details>

## Pour aller plus loin

- **[2.3d — Modèle gaussien, LDA et QDA](2.3d-Modele-Gaussien-LDA-QDA.ipynb)** : la même log-densité gaussienne, mais avec une **covariance jointe** estimée par classe. QDA est le modèle que Naive Bayes approche en supprimant les termes croisés — et c'est ce qui permet de mesurer exactement ce que l'hypothèse d'indépendance coûte.
- **[2.7 — Modèles non paramétriques](2.7-Modeles-Non-Parametriques.ipynb)** (et sa [version Rust](2.7-Modeles-Non-Parametriques-Rust.ipynb)) : quand la frontière n'est pas linéaire, ni le génératif ni le discriminatif linéaire ne suffisent ; le noyau prend le relais.
- **[2.3 — Régression linéaire et logistique](2.3-Regression-lineaire-logistique.ipynb)** : la logistique de ce carnet, dérivée depuis la vraisemblance et le gradient.
- **La version Python** de ce carnet ([2.3b](2.3b-Naive-Bayes-Generatif.ipynb)) et sa **version C#** ([2.3b C#](2.3b-Naive-Bayes-Generatif-CSharp.ipynb)) : mêmes scénarios, mêmes régimes, trois écosystèmes. Les niveaux absolus diffèrent d'un carnet à l'autre — les générateurs aussi.
- **La calibration** : un classifieur peut bien classer et mal estimer ses probabilités. La régression logistique est réputée mieux calibrée ; Naive Bayes avec des dimensions corrélées produit des probabilités trop tranchées. Un histogramme de fiabilité (reliability diagram) sur les deux modèles du scénario A est l'expérience naturelle à faire ensuite.

## Références

- Deisenroth, M. P., Faisal, A. A. & Ong, C. S. (2020). *Mathematics for Machine Learning*. Cambridge University Press — chapitre 12 (classification linéaire) pour la régression logistique, et les rappels de probabilité pour la règle de Bayes.
- Documentation de `smartcore` 0.7.0 : `naive_bayes::gaussian`, `naive_bayes::multinomial`, `linear::logistic_regression`.
- Documentation `scikit-learn` : `naive_bayes.GaussianNB` (paramètre `var_smoothing`) et `linear_model.LogisticRegression` (paramètre `C`), pour la correspondance des paramètres avec ce portage.